In [1]:
# Gather links with unstructured
# from unstructured.partition.html import partition_html
# cnn_lite_url = "https://lite.cnn.com/"
# elements = partition_html(url=cnn_lite_url)

# links = []

# for element in elements:
#     if element.metadata.link_urls:
#         relative_link = element.metadata.link_urls[0][1:]
#         if relative_link.startswith("2024"):
#             links.append(f"{cnn_lite_url}{relative_link}")

# print(f"We retrieved {len(links)} links to documents from {cnn_lite_url}")

In [2]:
# Ingest individual articles with Langchain UnstructuredURLLoader
# from langchain.document_loaders import UnstructuredURLLoader
# loaders = UnstructuredURLLoader(urls=links[:200], show_progress_bar=True)

# docs = loaders.load()
# #print(docs[0])
# print(f"We retrieved {len(docs)} links to documents from {cnn_lite_url}")

# # Variable docs is a list of langchain_core.documents.base.Document -> convert it to a list of strings
# docs_as_list_of_strings = [docs[i].page_content for i in range(len(docs))]

In [ ]:
# For reproducibility we can save and load the documents in a file
# -------------------------------------------------------
# Save the list of documents as a JSON file
# import json
# with open("docs_as_list_of_strings.json", "w") as file:
#     json.dump(docs_as_list_of_strings, file)

# Load the list of documents from a JSON file
import json
with open("docs_as_list_of_strings.json", "r") as file:
    docs_as_list_of_strings = json.load(file)
# -------------------------------------------------------

# Each document of the list is a very long text, so we split each document into smaller chuncks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

# Matrix where each row represent a document splitted in chunks (one in every column)
docs_as_list_of_chuncks = [] 
for item in docs_as_list_of_strings:
    docs_as_list_of_chuncks.append(text_splitter.split_text(item))

# Load ibm-granite/granite-guardian-hap-38m model
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model_name_or_path = 'ibm-granite/granite-guardian-hap-38m'
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

prediction_results = []
probability_results = []
for chunks in docs_as_list_of_chuncks:
    input = tokenizer(chunks, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**input).logits
        prediction = torch.argmax(logits, dim=1).detach().numpy().tolist() # Binary prediction where label 1 indicates toxicity.
        prediction_results.append(prediction)
        probability = torch.softmax(logits, dim=1).detach().numpy()[:,1].tolist() #  Probability of toxicity.
        probability_results.append(probability)


# Find the indices where the value is 1
import numpy as np
#indices = np.argwhere(prediction_results == 1)
indices = [(i, j) for i, row in enumerate(prediction_results) for j, value in enumerate(row) if value == 1]
# Print the results
for tup in indices:
    print(tup)  # This prints the whole tuple
    i=tup[0]; j=tup[1];
    #print(prediction_results[i][j])
    print(f"Probability: {probability_results[i][j]}, Sentence: {docs_as_list_of_chuncks[i][j]}")